# Delegation, inference, and scarce attention

This notebook regenerates every static figure and numerical diagnostic for the single-attempt model. A user chooses delegated scope $s$ and normalized model effort level $x\geq1$, where one is the minimum viable effort by choice of units. Every chunk consumes $sx$ token units and $h(s)$ review hours, whether it succeeds or fails; successful output has probability $P=qr$.

The work-limited objective is surplus per work unit, $u=bP-cx-wh/s$, with demand $D^W=WAx$. The attention-limited objective is surplus per review hour, $J=(s/h)(bP-cx)-w$, with demand $D^H=H(s/h)x$ conditional on operating. The purpose is to distinguish industry regimes and qualitative responses, not to forecast volumes.


## Setup

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from modeling_token_demand import (
    AttentionConstrainedOptimizer,
    HARD_EXECUTION,
    HIGH_ADOPTION_HURDLE,
    HIGH_CAPABILITY_REQUIREMENT,
    IndustryModel,
    LOW_INFERENCE_RETURNS,
    LOW_ADOPTION_HURDLE,
    OptimizationSettings,
    PolicyOptimizer,
    PROPORTIONAL_REVIEW,
    REFERENCE_INDUSTRY,
    Scenario,
    SLOW_REVIEW_GROWTH,
    illustrative_industries,
)

from modeling_token_demand.paradigms import gallery_settings, axis_values
from modeling_token_demand.calibrations import singleton_industries

FIGURE_DIR = ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
ATTENTION_HOURS_PER_INDUSTRY = 100_000.0
work_limited_industries = illustrative_industries()
industries = tuple(
    replace(industry, human_attention_hours=ATTENTION_HOURS_PER_INDUSTRY)
    for industry in illustrative_industries()
)

## Controlled one-parameter-at-a-time variations

Every plotted alternative starts from one reference industry and changes exactly one industry parameter. A systematic scan retains only cases that create a qualitatively different demand, adoption, or spending path. This makes the mechanism behind each line directly identifiable.

All cases use $b=w=100$, potential work $W=1{,}000{,}000$, and attention endowment $H=100{,}000$ where applicable. The baseline is $m=\eta=v=c=1$. The work-limited figures vary the hurdle location $\mu$, execution ease $a$, or capability horizon $\lambda$ one at a time; the attention-limited figures vary execution ease $a$, inference returns $\alpha$, or review elasticity $\beta$.

The reference hurdle location is $\mu=81$, close to the single-attempt reference surplus of about $80.3$. This keeps the reference near an adoption transition. The low- and high-hurdle cases use $\mu=40$ and $\mu=95$. Hard execution uses $a=1$, high capability requirement shortens $\lambda$ from 12 to 3, low inference returns sets $\alpha=0.2$, and the slow-growing and nearly proportional review cases set $\beta=0.15$ and $\beta=0.95$. Every alternative holds all other industry parameters fixed. No plotted level is an empirical industry estimate.

The following tables come directly from the calibration code. Plot lines are rows, industry parameters are columns, and bold cells are the only values that differ from the reference.


In [ ]:
from modeling_token_demand.calibrations import calibration_tables_markdown

reference = REFERENCE_INDUSTRY
comparison_by_name = {industry.name: industry for industry in work_limited_industries}
display(Markdown("Plot lines are rows; industry parameters are columns. Bold cells differ from the reference.\n\n" + calibration_tables_markdown()))
print("Eight active cases: one reference and seven one-parameter alternatives.")


## Numerical solution

Both optimizers search over $(\log s,\log x)$ using a coarse grid and multiple local refinements. The attention objective also has an independent scalar characterization. Wider bounds and denser searches check whether turning points are economic outcomes rather than numerical cutoffs.


In [ ]:
# Wider action bounds support the wider scenario ranges, not demand ceilings.
settings = gallery_settings()
per_work_optimizer = PolicyOptimizer(settings)
attention_optimizer = AttentionConstrainedOptimizer(settings)
baseline = Scenario()


In [ ]:
policy_cache = {}


def solve_axis(
    policy_optimizer, scenario_field, values, industry_set=industries
):
    """Reoptimize every supplied parameter case along one scenario axis."""
    results = {}
    for industry in industry_set:
        model = IndustryModel(industry)
        outcomes = []
        # Neither objective depends on adoption or the scale of the endowment.
        technical = replace(industry, name="", adoption_location=0, adoption_scale=1,
                            human_attention_hours=None)
        for value in values:
            scenario = replace(baseline, **{scenario_field: float(value)})
            key = (policy_optimizer, technical, scenario)
            if key not in policy_cache:
                policy_cache[key] = policy_optimizer.solve(model, scenario).policy
            outcomes.append(model.evaluate(policy_cache[key], scenario))
        results[industry.name] = outcomes
    return results


PRICE_LIMITS = (.1, 4.0)
PRICE_TICKS = (.1, .2, .5, 1, 2, 4)
CAPABILITY_TICKS = (.1, .25, 1, 2, 5, 10, 30)
EFFICIENCY_TICKS = (.25, .5, 1, 2, 5, 10)
GROUP_COLORS = {
    "Reference": "0.35",
    "Low adoption hurdle": "#1f77b4",
    "High adoption hurdle": "#ff7f0e",
    "Hard execution": "#2ca02c",
    "High capability requirement": "#d62728",
    "Low inference returns": "#9467bd",
    "Slow-growing review": "#8c564b",
    "Nearly proportional review": "#e377c2",
    "Capability constraint": "#0072B2",
    "Execution difficulty": "#E69F00",
    "Verification burden": "#009E73",
    "Economic value": "#CC79A7",
    "Adoption concentration": "#D55E00",
    "Early saturation": "#008B8B",
    "Capability valley": "#7A5195",
    "Offsetting efficiency": "#6B6B00",
}
GROUP_MARKERS = {
    "Low adoption hurdle": "o",
    "High adoption hurdle": "s",
    "Hard execution": "^",
    "High capability requirement": "x",
    "Low inference returns": "o",
    "Slow-growing review": "s",
    "Nearly proportional review": "x",
}


def variant_group_and_level(industry_name):
    """Return the controlled parameter group and high/low setting."""
    if industry_name == "Reference industry":
        return "Reference", "reference"
    if ": " not in industry_name:
        return industry_name, "singleton"
    group, level = industry_name.split(": ", maxsplit=1)
    return group, level


def variant_line_options(industry_name):
    """Use a solid gray reference and dashed, marked alternatives."""
    group, level = variant_group_and_level(industry_name)
    if level == "reference":
        return {
            "color": GROUP_COLORS[group], "linestyle": "-", "linewidth": 2.8,
            "zorder": 10,
        }
    return {
        "color": GROUP_COLORS[group],
        "linestyle": "--",
        "linewidth": 2.2 if level == "singleton" else 1.8,
        "marker": GROUP_MARKERS.get(group, "o"), "markevery": 9,
        "markersize": 5.5, "markerfacecolor": "white",
        "markeredgewidth": 1.0,
        "alpha": 0.95,
    }


def add_variant_legend(fig, ax, ncol=4, fontsize=8.4):
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="outside upper center", ncol=ncol,
        frameon=False, fontsize=fontsize, handlelength=3.2,
    )


def label_log_x_axis(ax, ticks):
    """Show informative numeric labels instead of sparse powers of ten."""
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{tick:g}" for tick in ticks])


def baseline_index(values, baseline_value):
    matches = np.flatnonzero(np.isclose(values, baseline_value))
    if len(matches) != 1:
        raise ValueError(f"expected one baseline value {baseline_value}")
    return int(matches[0])


def indexed_series(values, quantities, baseline_value):
    """Index a positive series to one at the common scenario baseline."""
    quantities = np.asarray(quantities, dtype=float)
    denominator = quantities[baseline_index(values, baseline_value)]
    if denominator <= 0:
        raise ValueError("cannot index a series with a nonpositive baseline")
    return quantities / denominator


def attention_limited_tokens(outcome):
    """Return demand under abundant work and a binding attention constraint."""
    if outcome.attention_limited_tokens is None:
        raise ValueError("human_attention_hours must be set for these plots")
    return outcome.attention_limited_tokens


REVENUE_BENCHMARK_LABEL = r"Constant revenue ($c_0/c$; price panel)"


def add_constant_revenue_benchmark(ax, prices, baseline_price):
    """Add demand needed to preserve each case's own baseline token revenue."""
    ax.plot(
        prices, baseline_price / np.asarray(prices, dtype=float),
        color="black", linestyle=":", linewidth=1.4, marker=None, zorder=12,
        label=REVENUE_BENCHMARK_LABEL,
    )
    ax.text(
        0.04, 0.95, "Above black line: higher revenue\nthan at the baseline price",
        transform=ax.transAxes, ha="left", va="top", fontsize=9,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85, "pad": 3},
        zorder=13,
    )


def plot_demand_panels(
    capabilities,
    capability_results,
    efficiencies,
    efficiency_results,
    prices,
    price_results,
    regime,
    normalize,
    filename,
):
    """Plot either resource regime in levels or within-case demand indexes."""
    regime_label, token_quantity = {
        "work": ("Work-limited", lambda outcome: outcome.work_limited_tokens),
        "attention": ("Attention-limited", attention_limited_tokens),
    }[regime]
    panels = (
        (
            capabilities, capability_results, "(a) Model capability",
            "Model capability, $m$", CAPABILITY_TICKS, 1.0, False,
        ),
        (
            efficiencies, efficiency_results, "(b) Token efficiency",
            r"Token efficiency, $\eta$", EFFICIENCY_TICKS, 1.0, False,
        ),
        (
            prices, price_results, "(c) Token price",
            "Normalized token price, $c$ (lower cost to the right)",
            PRICE_TICKS, 1.0, True,
        ),
    )
    mixed_work_axes = regime == "work"
    fig, axes = plt.subplots(1, 3, figsize=(16.0, 6.8), sharey=not mixed_work_axes)
    for ax, panel in zip(axes, panels):
        values, results, title, xlabel, ticks, baseline_value, reverse_x = panel
        for industry_name, outcomes in results.items():
            demand = np.array([
                token_quantity(outcome) for outcome in outcomes
            ])
            if normalize:
                demand = indexed_series(values, demand, baseline_value)
            else:
                demand = demand / 1e6
            ax.plot(
                values, demand, label=industry_name,
                **variant_line_options(industry_name),
            )
        ax.set_xscale("log")
        linear_y = mixed_work_axes and not reverse_x
        ax.set_yscale("linear" if linear_y else "log")
        if linear_y:
            ax.set_ylim(bottom=0)
        label_log_x_axis(ax, ticks)
        ax.axvline(
            baseline_value, color="0.55", linestyle=":", linewidth=1.1
        )
        if normalize:
            ax.axhline(1.0, color="0.72", linestyle=":", linewidth=0.9)
            if reverse_x:  # Only the price panel has a revenue benchmark.
                add_constant_revenue_benchmark(ax, values, baseline_value)
        if reverse_x:
            ax.set_xlim(PRICE_LIMITS[1], PRICE_LIMITS[0])
        ax.set_title(title, loc="left")
        ax.set_xlabel(xlabel)
        ax.grid(which="major", alpha=0.25)
        ax.grid(which="minor", alpha=0.08)

    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.99),
        ncol=4, frameon=False, fontsize=8.2, handlelength=3.2,
    )
    fig.subplots_adjust(
        left=0.07, right=0.99, bottom=0.13, top=0.74, wspace=0.22 if mixed_work_axes else 0.03
    )
    if normalize:
        fig.supylabel(f"{regime_label} demand index (baseline = 1)")
    else:
        fig.supylabel(f"{regime_label} token units (millions / period)")
    fig.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches="tight")
    return fig


def plot_demand(
    values, results, xlabel, filename, baseline_value, xticks, reverse_x=False,
    revenue_benchmark=False,
):
    """Plot indexed attention demand for the controlled parameter cases."""
    fig, ax = plt.subplots(figsize=(11.0, 7.0), constrained_layout=True)
    for industry_name, outcomes in results.items():
        demand = indexed_series(
            values,
            [attention_limited_tokens(outcome) for outcome in outcomes],
            baseline_value,
        )
        ax.plot(
            values, demand, label=industry_name,
            **variant_line_options(industry_name),
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    label_log_x_axis(ax, xticks)
    ax.axvline(baseline_value, color="0.55", linestyle=":", linewidth=1.1)
    ax.axhline(1.0, color="0.72", linestyle=":", linewidth=0.9)
    if revenue_benchmark:
        add_constant_revenue_benchmark(ax, values, baseline_value)
    if reverse_x:
        ax.set_xlim(PRICE_LIMITS[1], PRICE_LIMITS[0])
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Attention-limited token demand index (baseline = 1)")
    ax.grid(which="major", alpha=0.25)
    ax.grid(which="minor", alpha=0.10)
    add_variant_legend(fig, ax)
    fig.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches="tight")
    return fig


def plot_capability_objective_comparison(
    capabilities, attention_results, per_work_results, filename
):
    """Compare the two objectives for the reference industry."""
    name = "Reference industry"
    attention_demand = indexed_series(
        capabilities,
        [attention_limited_tokens(outcome) for outcome in attention_results[name]],
        1.0,
    )
    per_work_demand = indexed_series(
        capabilities,
        [attention_limited_tokens(outcome) for outcome in per_work_results[name]],
        1.0,
    )
    fig, ax = plt.subplots(figsize=(10.0, 6.2), constrained_layout=True)
    ax.plot(
        capabilities, attention_demand, color="#0072B2", linewidth=2.3,
        label="Optimize per attention-hour",
    )
    ax.plot(
        capabilities, per_work_demand, color="0.35", linestyle="--",
        linewidth=2.1, label="Optimize per work-hour, then cap",
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    label_log_x_axis(ax, CAPABILITY_TICKS)
    ax.axvline(1.0, color="0.55", linestyle=":", linewidth=1.0)
    ax.axhline(1.0, color="0.72", linestyle=":", linewidth=0.9)
    ax.grid(which="major", alpha=0.25)
    ax.grid(which="minor", alpha=0.08)
    ax.set_xlabel("Model capability, $m$")
    ax.set_ylabel("Attention-limited token demand index ($m=1$ = 1)")
    ax.legend(frameon=False)
    fig.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches="tight")
    return fig


def plot_token_spend(values, results, filename, xticks):
    """Plot indexed token spending as token price changes."""
    fig, ax = plt.subplots(figsize=(11.0, 7.0), constrained_layout=True)
    for industry_name, outcomes in results.items():
        spend = indexed_series(
            values,
            [
                attention_limited_tokens(outcome) * token_price
                for outcome, token_price in zip(outcomes, values)
            ],
            1.0,
        )
        ax.plot(
            values, spend, label=industry_name,
            **variant_line_options(industry_name),
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    label_log_x_axis(ax, xticks)
    ax.axvline(1.0, color="0.55", linestyle=":", linewidth=1.1)
    ax.axhline(1.0, color="0.72", linestyle=":", linewidth=0.9)
    ax.set_xlim(PRICE_LIMITS[1], PRICE_LIMITS[0])
    ax.set_xlabel("Normalized token price, $c$ (lower cost to the right)")
    ax.set_ylabel("Attention-limited token spend index (baseline = 1)")
    ax.grid(which="major", alpha=0.25)
    ax.grid(which="minor", alpha=0.10)
    add_variant_legend(fig, ax)
    fig.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches="tight")
    return fig


def plot_surplus_by_capability(capabilities, results, filename, xticks):
    """Plot indexed optimized per-work surplus as capability changes."""
    fig, ax = plt.subplots(figsize=(11.0, 7.0), constrained_layout=True)
    for industry_name, outcomes in results.items():
        surplus = indexed_series(
            capabilities,
            [outcome.surplus_per_work_hour for outcome in outcomes],
            1.0,
        )
        ax.plot(
            capabilities, surplus, label=industry_name,
            **variant_line_options(industry_name),
        )

    ax.set_xscale("log")
    label_log_x_axis(ax, xticks)
    ax.axvline(1.0, color="0.55", linestyle=":", linewidth=1.1)
    ax.axhline(1.0, color="0.72", linestyle=":", linewidth=0.9)
    ax.set_xlabel("Model capability, $m$")
    ax.set_ylabel("Optimized per-work surplus index ($m=1$ = 1)")
    ax.grid(which="major", alpha=0.25)
    ax.grid(which="minor", alpha=0.10)
    add_variant_legend(fig, ax)
    fig.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches="tight")
    return fig


def plot_attention_shadow_price(capabilities, results, filename):
    """Plot indexed shadow value and its exact capability elasticity."""
    fig, axes = plt.subplots(1, 2, figsize=(13.2, 6.5), constrained_layout=True)
    level_ax, elasticity_ax = axes
    industries_by_name = {industry.name: industry for industry in industries}

    for industry_name, outcomes in results.items():
        industry = industries_by_name[industry_name]
        shadow_prices = indexed_series(
            capabilities,
            [
                outcome.surplus_per_attention_hour + industry.human_cost_per_hour
                for outcome in outcomes
            ],
            1.0,
        )
        elasticities = []
        for outcome in outcomes:
            s = outcome.policy.delegation_hours
            review_growth = (1 + s) ** industry.verification_elasticity
            variable_review = industry.verification_scale * (review_growth - 1)
            theta = (
                industry.verification_elasticity * industry.verification_scale
                * s * review_growth
                / ((1 + s) * (industry.verification_fixed_hours + variable_review))
            )
            elasticities.append(1.0 - theta)

        plot_options = variant_line_options(industry_name)
        level_ax.plot(
            capabilities, shadow_prices, label=industry_name, **plot_options
        )
        elasticity_ax.plot(capabilities, elasticities, **plot_options)

    for ax in axes:
        ax.set_xscale("log")
        label_log_x_axis(ax, CAPABILITY_TICKS)
        ax.axvline(1.0, color="0.55", linestyle=":", linewidth=1.1)
        ax.grid(which="major", alpha=0.25)
        ax.grid(which="minor", alpha=0.08)
        ax.set_xlabel("Model capability, $m$")

    level_ax.set_yscale("log")
    level_ax.axhline(1.0, color="0.72", linestyle=":", linewidth=0.9)
    level_ax.set_ylabel("Shadow price index ($m=1$ = 1)")
    elasticity_ax.set_ylim(0.0, 1.02)
    elasticity_ax.set_ylabel(
        r"Exact elasticity, $d\log \rho^\star / d\log m$"
    )
    add_variant_legend(fig, level_ax, ncol=4, fontsize=8.1)
    fig.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches="tight")
    return fig


### Baseline attention-constrained policies

The table reports each case's scope, model effort level, success, value, and demand before indexing.


In [ ]:
baseline_rows = []
for industry in industries:
    outcome = attention_optimizer.solve(IndustryModel(industry), baseline)
    baseline_rows.append(
        f"| {industry.name} | {outcome.policy.delegation_hours:.3f} | "
        f"{outcome.policy.tokens_per_work_hour:,.0f} | "
        f"{outcome.success_probability:.1%} | "
        f"{outcome.surplus_per_attention_hour:,.0f} | "
        f"{attention_limited_tokens(outcome) / 1e6:.1f} |"
    )
baseline_header = (
    "| Parameter case | s* (hours) | x* (token units/hour) | Success | Surplus/attention-hour | Attention demand (millions of token units) |\n"
    "|---|---:|---:|---:|---:|---:|\n"
)
display(Markdown(baseline_header + "\n".join(baseline_rows)))

## Token price

The horizontal axis is reversed: moving right makes token units cheaper. Each curve reoptimizes $(s,x)$ and is indexed to its own demand at $c=1$. Revealed preference implies that a pure price reduction cannot reduce optimized token quantity in either polar regime. Revenue need not rise.


In [ ]:
token_prices = axis_values(.1, 8, 1, extra=[.2, .5, 2, 4])
price_results = solve_axis(
    attention_optimizer, "token_price", token_prices
)
work_limited_price_results = solve_axis(
    per_work_optimizer,
    "token_price",
    token_prices,
    industry_set=work_limited_industries,
)
price_figure = plot_demand(
    token_prices,
    price_results,
    xlabel="Normalized token price, $c$ (lower cost to the right)",
    filename="token-demand-vs-price.png",
    baseline_value=1.0,
    xticks=PRICE_TICKS,
    reverse_x=True,
    revenue_benchmark=True,
)
spend_figure = plot_token_spend(
    token_prices, price_results, filename="token-spend-vs-price.png",
    xticks=PRICE_TICKS,
)

The price panel's black line is the constant-revenue benchmark $c_0/c$. Above it, spending exceeds the same industry's spending at the baseline price. This comparison is relative to that baseline, not a statement about every additional price cut. The separate spending figure keeps revenue distinct from token quantity.


## Token efficiency

Higher $\eta$ raises effective inference per token unit without expanding the capability frontier. Writing $z=\eta x$ makes efficiency equivalent to a lower price for effective inference, followed by division of token demand by $\eta$. For unconstrained policies, $D(c,\eta)=D(c/\eta,1)/\eta$ in either regime.


In [ ]:
token_efficiencies = axis_values(.25, 10, 1, extra=[2, 5])
efficiency_results = solve_axis(
    attention_optimizer, "token_efficiency", token_efficiencies
)
work_limited_efficiency_results = solve_axis(
    per_work_optimizer,
    "token_efficiency",
    token_efficiencies,
    industry_set=work_limited_industries,
)
efficiency_figure = plot_demand(
    token_efficiencies,
    efficiency_results,
    xlabel=r"Token efficiency, $\eta$",
    filename="token-demand-vs-efficiency.png",
    baseline_value=1.0,
    xticks=EFFICIENCY_TICKS,
)

Efficiency saves token units for a given task policy, but can also increase adoption or work supervised per attention hour. Demand rises when the induced expansion more than offsets the saving. The concentrated-adoption case illustrates a hump as market expansion gives way to saturation; the offsetting-efficiency case illustrates nearly balanced effects over a finite range.


## Model capability

Capability $m$ increases both the solvable share $q$ and conditional execution probability $r$. At a fixed policy it raises $P=qr$ without refunding any token or review expenditure. Reoptimization can change scope, inference, and adoption, so higher optimized value does not imply higher token demand.


### Shadow price of scarce attention

Let $L=s/h(s)$ and define gross value before the attention charge as $R=L(bP-cx)$, so $J=R-w$. In the active attention-limited regime, $\rho^*=\max R$ values an additional review hour before charging $w$; the net capacity shadow value is $\rho^*-w$.

At an interior optimum, the envelope theorem and scope first-order condition imply

$$
\frac{d\log\rho^*}{d\log m}=1-\theta(s^*),\qquad
\theta(s)=\frac{\beta h_1s(1+s)^{\beta-1}}{h_0+h_1[(1+s)^\beta-1]}.
$$

For sublinear verification, the value of attention rises even when token demand falls. The scalar attention solver checks this identity independently of the general search.


In [ ]:
model_capabilities = axis_values(.1, 30, 1, extra=[.25, .5, .8, 2, 5, 10])
attention_capability_results = solve_axis(
    attention_optimizer, "model_capability", model_capabilities
)
per_work_capability_results = solve_axis(
    per_work_optimizer, "model_capability", model_capabilities,
    industry_set=tuple(
        replace(industry, human_attention_hours=ATTENTION_HOURS_PER_INDUSTRY)
        for industry in work_limited_industries
    ),
)
work_limited_capability_results = solve_axis(
    per_work_optimizer,
    "model_capability",
    model_capabilities,
    industry_set=work_limited_industries,
)
capability_figure = plot_demand(
    model_capabilities,
    attention_capability_results,
    xlabel="Model capability, m",
    filename="token-demand-vs-capability.png",
    baseline_value=1.0,
    xticks=CAPABILITY_TICKS,
)
capability_comparison_figure = plot_capability_objective_comparison(
    model_capabilities,
    attention_capability_results,
    per_work_capability_results,
    filename="token-demand-vs-capability-objectives.png",
)

In [ ]:
exact_attention_capability_results = {}
for industry in industries:
    model = IndustryModel(industry)
    exact_attention_capability_results[industry.name] = [
        attention_optimizer.solve_interior(
            model, replace(baseline, model_capability=float(capability))
        )
        for capability in model_capabilities
    ]

objective_differences = []
for industry in industries:
    name = industry.name
    for general, scalar in zip(
        attention_capability_results[name], exact_attention_capability_results[name]
    ):
        objective_differences.append(
            abs(general.surplus_per_attention_hour - scalar.surplus_per_attention_hour)
            / (abs(scalar.surplus_per_attention_hour) + industry.human_cost_per_hour)
        )
print(
    "Largest relative difference between general and scalar solutions: "
    f"{max(objective_differences):.2e}"
)

shadow_price_figure = plot_attention_shadow_price(
    model_capabilities,
    exact_attention_capability_results,
    filename="shadow-price-of-attention-vs-capability.png",
)

shadow_rows = []
for industry in industries:
    model = IndustryModel(industry)
    endpoint_outcomes = [
        attention_optimizer.solve_interior(
            model, replace(baseline, model_capability=capability)
        )
        for capability in (.1, 1.0, 30.0)
    ]
    shadow_prices = [
        outcome.surplus_per_attention_hour + industry.human_cost_per_hour
        for outcome in endpoint_outcomes
    ]
    baseline_outcome = endpoint_outcomes[1]
    s = baseline_outcome.policy.delegation_hours
    review_growth = (1 + s) ** industry.verification_elasticity
    variable_review = industry.verification_scale * (review_growth - 1)
    elasticity = 1.0 - (
        industry.verification_elasticity * industry.verification_scale
        * s * review_growth
        / ((1 + s) * (industry.verification_fixed_hours + variable_review))
    )
    shadow_rows.append(
        f"| {industry.name} | {shadow_prices[0]:,.0f} | "
        f"{shadow_prices[1]:,.0f} | {shadow_prices[2]:,.0f} | "
        f"{elasticity:.3f} |"
    )
shadow_header = (
    "| Parameter case | $\\rho^*(0.1)$ | $\\rho^*(1)$ | "
    "$\\rho^*(30)$ | Elasticity at $m=1$ |\n"
    "|---|---:|---:|---:|---:|\n"
)
display(Markdown(shadow_header + "\n".join(shadow_rows)))


The left panel indexes gross attention value at $m=1$; the right panel gives its local capability elasticity. Neither panel measures token demand. The opportunity cost $w$ affects participation but is constant in the conditional attention objective.


The objective-comparison figure uses only the reference industry. The dashed curve first maximizes surplus per work unit and then applies an attention cap. It is a behavioral comparison, not the attention-optimal allocation. Each curve has its own baseline denominator, so compare changes rather than absolute levels.


### Per-work surplus as capability improves

This figure reports $\max_{s,x}u$. Higher capability improves a feasible policy's success without changing its cost, so optimized surplus cannot fall.


In [ ]:
surplus_capability_figure = plot_surplus_by_capability(
    model_capabilities,
    per_work_capability_results,
    filename="optimized-surplus-vs-model-capability.png",
    xticks=CAPABILITY_TICKS,
)

### Work-limited token demand and adoption

Potential work $W$ is fixed. Adoption follows a logistic distribution of hurdles, which may be positive or negative; the work-weighted share is $A=F(u^*)$. Assigned work is $WA$, completed work is $WAP$, and token demand is $WAx$. Larger chunks do not mechanically increase the amount of underlying work.

Five cases appear: one reference and four alternatives that each change one parameter. Capability and efficiency use independent linear y-axes; price uses an independent logarithmic y-axis so tiny adoption tails do not compress the other responses.


In [ ]:
work_limited_levels_figure = plot_demand_panels(
    model_capabilities,
    work_limited_capability_results,
    token_efficiencies,
    work_limited_efficiency_results,
    token_prices,
    work_limited_price_results,
    regime="work",
    normalize=False,
    filename="work-limited-token-demand-levels.png",
)
work_limited_indexed_figure = plot_demand_panels(
    model_capabilities,
    work_limited_capability_results,
    token_efficiencies,
    work_limited_efficiency_results,
    token_prices,
    work_limited_price_results,
    regime="work",
    normalize=True,
    filename="work-limited-token-demand-indexed.png",
)


Levels show quantities under the common illustrative endowment; indexes compare each case with itself at the scenario baseline. The price panel's black line is the common constant-revenue benchmark. Near-zero baseline adoption can produce large indexes, so read these together with adoption and level plots.


### Attention-limited token demand: levels and indexed changes

These panels reoptimize $J$ with abundant useful work and fixed review capacity. Demand is $H(s/h)x$, and successful output is $H(s/h)P$. Failure consumes attention even though it produces no output.

The adoption-hurdle alternatives coincide with the reference in this polar regime because adoption does not enter the attention-limited objective. Shared logarithmic y-scales permit cross-panel comparisons; independent scales reveal smaller changes.


In [ ]:
attention_limited_levels_figure = plot_demand_panels(
    model_capabilities,
    attention_capability_results,
    token_efficiencies,
    efficiency_results,
    token_prices,
    price_results,
    regime="attention",
    normalize=False,
    filename="attention-limited-token-demand-levels.png",
)
attention_limited_indexed_figure = plot_demand_panels(
    model_capabilities,
    attention_capability_results,
    token_efficiencies,
    efficiency_results,
    token_prices,
    price_results,
    regime="attention",
    normalize=True,
    filename="attention-limited-token-demand-indexed.png",
)

capability_baseline_index = baseline_index(model_capabilities, 1.0)
for name in (
    "Reference industry", "Hard execution", "Low inference returns",
    "Slow-growing review", "Nearly proportional review"
):
    outcomes = attention_capability_results[name]
    ratio = (
        attention_limited_tokens(outcomes[baseline_index(model_capabilities, 5.0)])
        / attention_limited_tokens(outcomes[capability_baseline_index])
    )
    print(f"{name}: demand at m=5 / demand at m=1 = {ratio:.3f}")


The shifted review function is not homogeneous in scope, so the earlier power-law scaling special case no longer applies. Positive fixed review overhead keeps the attention optimum interior. The capability-valley example has a smooth fall followed by a rise as the balance between inference savings and supervisory scope changes.


## Failure, expenditure, and completed work

The same costs are paid on successful and unsuccessful chunks. These reference policies illustrate that a lower success rate reduces useful output rather than creating free capacity. The attention-limited throughput formula is conditional on operating; a user with nonpositive optimized net value can choose not to operate.


In [ ]:
failure_rows = []
for label, optimizer in (("Fixed work", per_work_optimizer), ("Scarce attention", attention_optimizer)):
    industry = replace(REFERENCE_INDUSTRY, human_attention_hours=ATTENTION_HOURS_PER_INDUSTRY)
    outcome = optimizer.solve(IndustryModel(industry), baseline)
    policy = outcome.policy
    failure_rows.append(
        f"| {label} | {outcome.success_probability:.1%} | "
        f"{policy.tokens_per_work_hour:,.0f} | "
        f"{outcome.verification_hours_per_chunk / policy.delegation_hours:.3f} | "
        f"{outcome.cost_per_work_hour:.2f} |"
    )
failure_header = (
    "| Resource regime | Success P | Tokens/work unit | Review hours/work unit | Cost/work unit (USD) |\n"
    "|---|---:|---:|---:|---:|\n"
)
display(Markdown(failure_header + "\n".join(failure_rows)))


### Optimizer audit

Every plotted policy is checked for positive finite quantities and continuous-bound hits. Price-demand monotonicity and the scalar attention solution provide independent checks. Endpoints, baselines, and extrema are then resolved on a denser grid with tenfold wider bounds. Technical comparison labels and adoption-CDF crossings are checked at the actual policies.


In [ ]:
sweep_sets = {
    "attention price": price_results,
    "attention efficiency": efficiency_results,
    "attention capability": attention_capability_results,
    "work price": work_limited_price_results,
    "work efficiency": work_limited_efficiency_results,
    "work capability": work_limited_capability_results,
}
boundary_hits = []
policy_count = 0
for sweep_name, results in sweep_sets.items():
    for industry_name, outcomes in results.items():
        for point_index, outcome in enumerate(outcomes):
            policy_count += 1
            policy = outcome.policy
            for variable, value, lower, upper, check_lower in (
                ("s", policy.delegation_hours,
                 settings.min_delegation_hours, settings.max_delegation_hours, True),
                ("x", policy.tokens_per_work_hour,
                 settings.min_tokens_per_work_hour, settings.max_tokens_per_work_hour, False),
            ):
                if (check_lower and np.isclose(value, lower, rtol=1e-5)) or np.isclose(value, upper, rtol=1e-5):
                    boundary_hits.append((sweep_name, industry_name, point_index, variable))
            if sweep_name.startswith("attention"):
                assert outcome.surplus_per_attention_hour > 0
            assert np.isfinite(outcome.work_limited_tokens)
            assert outcome.work_limited_tokens > 0
assert not boundary_hits, boundary_hits
assert max(objective_differences) < 1e-8

# Price is increasing in these arrays, so optimized token quantity must not rise.
for results, quantity in (
    (price_results, attention_limited_tokens),
    (work_limited_price_results, lambda item: item.work_limited_tokens),
):
    for industry_name, outcomes in results.items():
        values = np.array([quantity(outcome) for outcome in outcomes])
        assert np.max(np.diff(values)) <= 1e-6 * np.max(values), industry_name

reference_work = per_work_optimizer.solve(IndustryModel(REFERENCE_INDUSTRY), baseline)
print(f"Audited {policy_count} policies; no numerical policy bounds bind.")
print(f"Reference work-limited adoption at baseline: {reference_work.adoption_share:.2%}.")
print("Price-demand monotonicity and scalar-solver agreement checks passed.")


# Audit the identifying restriction: every active alternative changes one
# and only one industry parameter from the reference.
from dataclasses import asdict

expected_changes = {
    "Low adoption hurdle": {"name", "adoption_location"},
    "High adoption hurdle": {"name", "adoption_location"},
    "Hard execution": {"name", "execution_scale"},
    "High capability requirement": {"name", "capability_horizon_hours"},
    "Low inference returns": {"name", "inference_returns"},
    "Slow-growing review": {"name", "verification_elasticity"},
    "Nearly proportional review": {"name", "verification_elasticity"},
}
reference_parameters = asdict(REFERENCE_INDUSTRY)
for industry in work_limited_industries:
    if industry == REFERENCE_INDUSTRY:
        continue
    changed = {
        field for field, value in asdict(industry).items()
        if value != reference_parameters[field]
    }
    assert changed == expected_changes[industry.name], (industry.name, changed)
assert LOW_ADOPTION_HURDLE.adoption_location < REFERENCE_INDUSTRY.adoption_location
assert HIGH_ADOPTION_HURDLE.adoption_location > REFERENCE_INDUSTRY.adoption_location
assert all(industry.human_cost_per_hour == 100.0 for industry in industries)
assert [industry.name for industry in industries] == [
    industry.name for industry in work_limited_industries
]
print("One-parameter identifying restriction passed for all seven alternatives.")

scenario_axes = (
    ("price", "token_price", token_prices),
    ("efficiency", "token_efficiency", token_efficiencies),
    ("capability", "model_capability", model_capabilities),
)
from modeling_token_demand.paradigms import audit_main_sweeps
main_report = audit_main_sweeps(settings, sweep_sets, scenario_axes, industries)
print(f"Full-comparison audit: {main_report['audit']}")


## Focused views of the Appendix B configurations

The focused figures select cases from the same one-parameter tables. Work views separate adoption takeoff, saturation, demand, and revenue. Attention views isolate the effects of hard execution and nearly proportional review. All lines join optimized samples without curve fitting; turning-point locations are sampled rather than exact thresholds. The diagnostics combine these focused views with the main sweep audits.


In [ ]:
import json
from IPython.display import Image
from modeling_token_demand.paradigms import build_paradigm_figures

paradigm_report = build_paradigm_figures(FIGURE_DIR)
paradigm_report["main"] = main_report
(FIGURE_DIR / "paradigms.json").write_text(json.dumps(paradigm_report, separators=(",", ":"), allow_nan=False))
display(Markdown("Appendix B figures use one shared configuration set; all numerical audits passed."))
for gallery_name in ("paradigm-work-demand.png", "paradigm-adoption-and-revenue.png", "paradigm-attention-capability.png"):
    display(Image(filename=str(FIGURE_DIR / gallery_name)))


## Supplementary interventions: which improvements change demand and automation?

Five controlled experiments separate token quantity, delegated work, and expected completed work under the two resource constraints:

1. Uniformly faster review changes $v$ and exactly scales active attention throughput by $1/v$ without changing policy.
2. Slower review growth changes $\beta$ and reduces review time at every positive scope.
3. Expanded feasibility changes $\lambda$ alone; higher $m$ also improves execution.
4. A hundredfold efficiency gain changes $\eta$ at three inference-return settings.
5. Higher marginal inference returns change $\alpha$ directly at the reference, higher-capability, and hard-execution settings.

Top rows hold potential work fixed; lower rows hold attention fixed. Work shares are percentages of potential work. Other outcomes are indexed to their own experiment baseline. Every comparison uses the single-attempt model and preserves token and attention costs on failures.


In [ ]:
from modeling_token_demand.interventions import build_intervention_figures

intervention_report = build_intervention_figures(FIGURE_DIR)
for question in ("verification-speed", "review-growth", "harness-feasibility", "efficiency-returns", "inference-returns"):
    display(Image(filename=str(FIGURE_DIR / f"intervention-{question}.png")))


## Main-text figures: show the economic margins

These eleven views use the audited results above. Eight regime figures show adoption, supervisory leverage, or the hourly reservation price of scarce review attention beside demand, spending, or the fully reoptimized reservation token price. The three capability-lever figures put review elasticity, review cost, and inference returns directly on the horizontal axis and compare adoption with revenue. The complete gallery remains available for comparison.


In [ ]:
from modeling_token_demand.paper_figures import build_exposition_figures

exposition_figures = build_exposition_figures(FIGURE_DIR, paradigm_report, intervention_report)
for filename in exposition_figures:
    display(Image(filename=str(FIGURE_DIR / filename)))
